# 02. エピトープシグネチャ類似度マップ（UMAP）

**やること**：1st paper の glycogene UMAP（`1st_paper/UMAP/UMAP_glycogenes.R`：点＝サンプル、色＝薬、形＝細胞株）
と同じ枠を、**エピトープ空間**で描く。「肝でA薬、肺でB薬が近い」をマップを見て読む。

**スタンス**：まず描く。検証設計は図を見てから決める（合格条件を先に決めない）。

## 3つの層を並べて比較

| 層 | 次元 | 位置づけ |
|---|---|---|
| (a) ランドマーク | 959 | **実測**。処理群で取れる最大幅（全23,614遺伝子は対照サンプルにしか無い） |
| (b) glycogene | 385 | 推論。1st paper と直接対応する層 |
| (c) epitope | ~39 | 推論＋経路射影。3rd paper の寄与 |

## 前処理を2通り

- **センタリングなし**（Level 5 のまま）— Level 5 は既にプレート集団に対するz化済み。細胞株で固まるはず
- **細胞内センタリング**（薬剤平均を細胞ごとに引く）— 細胞の素性を除き薬剤効果を見る

「細胞で固まるか薬で固まるか」自体が読みたい情報なので両方出す。

## 目印（合格条件ではなく地図を読むための旗）

- `tunicamycin` — N型糖鎖合成阻害（DPAGT1）。23細胞株に存在
- `brefeldin-a` — ER-Golgi輸送阻害。86細胞株
- `monensin` — Golgi ionophore
- 系統一致の承認薬（MCF7乳6 / PC3・VCAP前立腺3 / A549・HCC515肺2 / A375メラノーマ1 / HT29大腸1 / HEPG2肝0）

In [ ]:
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import snowflake.connector as sc
from umap import UMAP
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = ['Hiragino Sans', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['svg.fonttype'] = 'none'

SEED = 42
FIGDIR, TABDIR = '../../results/figures', '../../results/tables'
CELLS = ['HEPG2', 'HUH7', 'PHH', 'A549', 'HCC515', 'MCF7', 'PC3', 'VCAP', 'A375', 'HT29']
DOSE_G, DOSE_L, TP = '10 uM', '10 µM', '6 h'     # 表記がテーブル間で異なる
POSCTL = ['tunicamycin', 'brefeldin-a', 'monensin']

con = sc.connect(account='DUETMBM-LL33279', user='KOREEDA', role='ACCOUNTADMIN',
                 warehouse='BIOINFORMATICS_XS', authenticator='SNOWFLAKE_JWT',
                 private_key_file=os.path.expanduser('~/.ssh/snowflake_rsa_key.pem'))
cur = con.cursor()
norm = lambda g: str(g).upper().replace('-', '_')
print('connected')

## 1. 承認薬リストとアンカー定義

In [ ]:
ind = cur.execute("SELECT COMPOUND_ID, MESH_HEADING, MAX_PHASE_FOR_IND "
                  "FROM RAW.LINCS.COMPOUND_TO_INDICATION").fetch_pandas_all()
ind.columns = [c.lower() for c in ind.columns]
ind['cid'] = ind['compound_id'].str.upper()
ind = ind[pd.to_numeric(ind['max_phase_for_ind'], errors='coerce') >= 4]

ONC_PAT = r'neoplas|carcinoma|leukemia|lymphoma|melanoma|myeloma|sarcoma'
approved = set(ind['cid'])
onc = set(ind.loc[ind['mesh_heading'].str.lower().str.contains(ONC_PAT, na=False), 'cid'])
LINEAGE = {'MCF7': ('breast', 'Breast Neoplasms'), 'PC3': ('prostate', 'Prostatic Neoplasms'),
           'VCAP': ('prostate', 'Prostatic Neoplasms'), 'A549': ('lung', 'Carcinoma, Non-Small-Cell Lung'),
           'HCC515': ('lung', 'Carcinoma, Non-Small-Cell Lung'), 'HT29': ('colorectal', 'Colorectal Neoplasms'),
           'A375': ('melanoma', 'Melanoma'), 'HEPG2': ('liver', 'Liver Neoplasms'),
           'HUH7': ('liver', 'Liver Neoplasms'), 'PHH': ('liver_normal', None)}
flag = {c: (set(ind.loc[ind['mesh_heading'] == m, 'cid']) if m else set()) for c, (_, m) in LINEAGE.items()}

print(f'承認薬 {len(approved)} / がん適応 {len(onc)} / 転用候補の母集団 {len(approved)-len(onc)}')
print('※ 地図は全化合物で作り、読み出し時に承認薬で絞る')

ann = pd.read_csv('/Users/koreedatatsuya/research/lincs_glyco_2nd_paper/inputs/'
                  'moa_target_master/compound_annotation_master.csv')
ann['cid'] = ann['compound_id'].str.upper()
ann = ann[~ann.cid.duplicated()].set_index('cid')
print('annotation:', ann.shape)

## 2. 3つの層のサンプル行列

用量・時間は `10 uM / 6 h` に固定（各細胞で最も承認薬カバレッジが高い共通条件）。

In [ ]:
def fetch_wide(table, gene_filter=None, dose=DOSE_G, meta_lower=True):
    cols = [r[0] for r in cur.execute(
        f"SELECT COLUMN_NAME FROM RAW.INFORMATION_SCHEMA.COLUMNS "
        f"WHERE TABLE_SCHEMA='LINCS' AND TABLE_NAME='{table}'").fetchall()]
    return cols

# --- (a) 実測ランドマーク 959 ---
t = time.time()
lm = cur.execute(
    f"SELECT * FROM RAW.LINCS.L1000_LEVEL5_LANDMARK "
    f"WHERE PERT_TYPE='trt_cp' AND UPPER(CELL) IN ({','.join(repr(c) for c in CELLS)}) "
    f"AND DOSE='{DOSE_L}' AND TIMEPOINT='{TP}'").fetch_pandas_all()
LMETA = {'PERT_TYPE', 'DOSE', 'PERTID', 'SAMPLE_ID', 'PERTNAME', 'TIMEPOINT', 'CELL'}
lm_genes = [c for c in lm.columns if c.upper() not in LMETA]
lm[lm_genes] = lm[lm_genes].apply(pd.to_numeric, errors='coerce')
lm['cell'] = lm['CELL'].str.upper(); lm['drug'] = lm['PERTNAME']
A = lm.groupby(['cell', 'drug'])[lm_genes].mean()
print(f'(a) landmark {A.shape}  ({time.time()-t:.0f}s)')

In [ ]:
# --- (b) glycogene 385 / (c) epitope ---
steps = cur.execute('SELECT epitope_id,epitope_name,step_id,hgnc_symbol '
                    'FROM RAW.GLYCOEPITOPE.EPITOPE_STEP_GENE').fetch_pandas_all()
steps.columns = [c.lower() for c in steps.columns]
steps['hgnc_symbol'] = steps['hgnc_symbol'].map(norm)

gcols = set(r[0] for r in cur.execute(
    "SELECT COLUMN_NAME FROM RAW.INFORMATION_SCHEMA.COLUMNS "
    "WHERE TABLE_SCHEMA='LINCS' AND TABLE_NAME='GLYCO_GENES_WIDE'").fetchall())
orig = {norm(c): c for c in gcols}
GMETA = {'VALUE','CANONICAL_SMILES','CELL','CMAPID','COMPOUND_ALIAS','DOSE','INCHI_KEY',
         'PERTID','PERTNAME','TIMEPOINT','SAMPLE_ID'}
gl_genes = sorted({norm(c) for c in gcols} - GMETA)

sel = ', '.join(['"cell"', '"pertname"'] + [f'"{orig[g]}"' for g in gl_genes])
gw = cur.execute(f'SELECT {sel} FROM RAW.LINCS.GLYCO_GENES_WIDE '
                 f'WHERE "cell" IN ({",".join(repr(c) for c in CELLS)}) '
                 f'AND "dose"=%s AND "timepoint"=%s', (DOSE_G, TP)).fetch_pandas_all()
gw.columns = ['cell', 'drug'] + gl_genes
gw[gl_genes] = gw[gl_genes].apply(pd.to_numeric, errors='coerce')
B = gw.groupby(['cell', 'drug'])[gl_genes].mean()
print(f'(b) glycogene {B.shape}')

# epitope射影（step内max × step間min）
gidx = {g: i for i, g in enumerate(gl_genes)}
def project(M):
    out, names = [], []
    for (epid, epname), grp in steps.groupby(['epitope_id', 'epitope_name']):
        sv = []
        for _, sg in grp.groupby('step_id'):
            ix = [gidx[g] for g in sg['hgnc_symbol'].unique() if g in gidx]
            if ix: sv.append(np.nanmax(M[:, ix], axis=1))
        if sv:
            out.append(np.nanmin(np.vstack(sv), axis=0)); names.append(f'{epid}|{epname}')
    return pd.DataFrame(np.array(out).T, columns=names)

Cc = project(B.to_numpy(dtype=float)); Cc.index = B.index
Cc = Cc.dropna(axis=1, how='any')
Cc = Cc.loc[:, ~Cc.T.round(6).duplicated()]
print(f'(c) epitope {Cc.shape}（重複列を縮約）')

LAYERS = {'(a) landmark 959 [実測]': A, '(b) glycogene 385 [推論]': B, '(c) epitope [経路射影]': Cc}
for k, v in LAYERS.items():
    print(f'{k:<28} {v.shape}  細胞別 {v.groupby(level=0).size().to_dict()}')

## 3. UMAP（1st paper に合わせて euclidean）

In [ ]:
def embed(X, center_by_cell):
    X = X.dropna(axis=1, how='any')
    if center_by_cell:
        X = X - X.groupby(level=0).transform('mean')
    Z = UMAP(n_neighbors=15, min_dist=0.1, n_components=2,
             metric='euclidean', random_state=SEED).fit_transform(X.values)
    return pd.DataFrame(Z, columns=['UMAP1', 'UMAP2'], index=X.index)

EMB = {}
for name, X in LAYERS.items():
    for ctr in (False, True):
        t = time.time()
        EMB[(name, ctr)] = embed(X, ctr)
        print(f'{name:<28} center={str(ctr):<5} {time.time()-t:.0f}s')

## 4. マップを描く

形＝細胞株、色＝薬の属性（1st paper の意匠）。目印の化合物にはラベルを付ける。

In [ ]:
MARKERS = ['o','s','^','D','v','P','X','*','<','>']
CELL_MK = {c: MARKERS[i % len(MARKERS)] for i, c in enumerate(CELLS)}

def color_series(idx, mode):
    drugs = idx.get_level_values('drug').str.upper()
    cells = idx.get_level_values('cell')
    if mode == 'onc':
        s = pd.Series('未承認/その他', index=range(len(idx)))
    s[np.isin(drugs, list(approved))] = 'その他の承認薬'
        s[np.isin(drugs, list(onc))] = 'がん適応薬'
        s[[d.lower() in POSCTL for d in idx.get_level_values('drug')]] = '糖鎖阻害剤(目印)'
        s[[d in flag.get(c, set()) for c, d in zip(cells, drugs)]] = '系統一致の承認薬(旗)'
        return s.values
    if mode == 'atc':
        return pd.Series(drugs).map(ann['ATC_L1_desc']).fillna('不明').values
    return np.array(['—'] * len(idx))

def draw(ax, E, mode, title):
    cvals = color_series(E.index, mode)
    cats = [c for c in pd.unique(cvals) if c is not None]
    if mode == 'atc':
        top = pd.Series(cvals).value_counts().head(9).index.tolist()
        cvals = np.array([c if c in top else 'その他' for c in cvals]); cats = top + ['その他']
    pal = plt.cm.tab10(np.linspace(0, 1, max(len(cats), 2)))
    cmap = dict(zip(cats, pal))
    if mode == 'onc':
        cmap.update({'未承認/その他': (.80,.82,.84,.12), 'その他の承認薬': (.35,.55,.70,.55), 'がん適応薬': (.10,.35,.65,.95),
                     '系統一致の承認薬(旗)': (.75,.22,.17,1.), '糖鎖阻害剤(目印)': (.1,.6,.25,1.)})
        order = ['未承認/その他', 'その他の承認薬', 'がん適応薬', '系統一致の承認薬(旗)', '糖鎖阻害剤(目印)']
        cats = [c for c in order if c in cats]
    for cell in CELLS:
        m = E.index.get_level_values('cell') == cell
        if not m.any(): continue
        for cat in cats:
            mm = m & (cvals == cat)
            if not mm.any(): continue
            big = cat in ('系統一致の承認薬(旗)', '糖鎖阻害剤(目印)')
            ax.scatter(E.UMAP1[mm], E.UMAP2[mm], marker=CELL_MK[cell],
                       s=46 if big else 3.5, color=cmap[cat],
                       lw=.5 if big else 0, edgecolor='k' if big else 'none', zorder=3 if big else 1)
    for (cell, drug) in E.index:
        if drug.lower() in POSCTL:
            ax.annotate(f'{drug}\n{cell}', (E.loc[(cell, drug), 'UMAP1'], E.loc[(cell, drug), 'UMAP2']),
                        fontsize=5.5, alpha=.85)
    ax.set_title(title, fontsize=10, loc='left')
    ax.set_xlabel('UMAP1'); ax.set_ylabel('UMAP2')
    return cats, cmap

for mode, tag in (('onc', 'anchor'), ('atc', 'atc')):
    fig, axes = plt.subplots(2, 3, figsize=(19, 12))
    for j, name in enumerate(LAYERS):
        for i, ctr in enumerate((False, True)):
            cats, cmap = draw(axes[i, j], EMB[(name, ctr)], mode,
                              f'{name}\n{"細胞内センタリングあり" if ctr else "センタリングなし (Level5のまま)"}')
    h1 = [plt.Line2D([], [], marker='o', ls='', color=cmap[c], label=c, mec='k', mew=.4) for c in cats]
    h2 = [plt.Line2D([], [], marker=CELL_MK[c], ls='', color='#555', label=c) for c in CELLS]
    fig.legend(handles=h1 + h2, loc='lower center', ncol=8, fontsize=8, frameon=False)
    fig.suptitle('エピトープシグネチャ類似度マップ — 承認薬 × 10細胞株, 10 uM / 6 h', fontsize=13)
    plt.tight_layout(rect=[0, .06, 1, .97])
    for ext in ('png', 'pdf', 'svg'):
        fig.savefig(f'{FIGDIR}/fig_epitope_umap_{tag}.{ext}', dpi=300, bbox_inches='tight')
    print('saved', tag)

## 5. 旗の近傍に何が来るか（元の問いの答え）

In [ ]:
def neighbors_of_flags(E, k=8):
    rows = []
    for (cell, drug) in E.index:
        if drug.upper() not in flag.get(cell, set()):
            continue
        d = np.linalg.norm(E.values - E.loc[(cell, drug)].values, axis=1)
        near = pd.Series(d, index=E.index).drop((cell, drug)).nsmallest(k)
        for (c2, d2), dist in near.items():
            rows.append({'flag_cell': cell, 'flag_drug': drug, 'nb_cell': c2, 'nb_drug': d2,
                         'dist': round(dist, 3), 'cross_cell': c2 != cell,
                         'nb_moa': ann['MOA'].get(d2.upper(), None)})
    return pd.DataFrame(rows)

nb = neighbors_of_flags(EMB[('(c) epitope [経路射影]', True)])
print(f'旗 {nb[["flag_cell","flag_drug"]].drop_duplicates().shape[0]} 本、近傍 {len(nb)} 件')
print(f'うち別細胞から来た近傍 {nb.cross_cell.sum()} 件')
nb[nb.cross_cell].head(25)

## 6. 強度フィルタ — ノイズ点を落とす

10 uM / 6 h の承認薬の多くはほとんど応答を起こしておらず、UMAPがノイズに支配される。
各細胞内でシグネチャ強度（L2ノルム）上位に絞り、地図を描き直す。

In [ ]:
STRENGTH_Q = 0.5   # 各細胞で強度上位50%を残す

def strength_filter(X, q=STRENGTH_Q):
    X = X.dropna(axis=1, how='any')
    Xc = X - X.groupby(level=0).transform('mean')
    mag = pd.Series(np.linalg.norm(Xc.values, axis=1), index=X.index)
    keep = mag.groupby(level=0).transform(lambda s: s >= s.quantile(1 - q))
    # 目印の化合物は強度によらず残す
    ispos = [d.lower() in POSCTL for _, d in X.index]
    return X[keep.values | np.array(ispos)], mag

FILT, MAG = {}, {}
for name, X in LAYERS.items():
    FILT[name], MAG[name] = strength_filter(X)
    print(f'{name:<28} {X.shape[0]} -> {FILT[name].shape[0]} 点')

EMB_F = {}
for name, X in FILT.items():
    EMB_F[name] = embed(X, center_by_cell=True)
print('embedded')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 6))
for j, name in enumerate(FILT):
    cats, cmap = draw(axes[j], EMB_F[name], 'onc', f'{name}\n強度上位{int(STRENGTH_Q*100)}% + 細胞内センタリング')
h1 = [plt.Line2D([], [], marker='o', ls='', color=cmap[c], label=c, mec='k', mew=.4) for c in cats]
h2 = [plt.Line2D([], [], marker=CELL_MK[c], ls='', color='#555', label=c) for c in CELLS]
fig.legend(handles=h1 + h2, loc='lower center', ncol=8, fontsize=8, frameon=False)
fig.suptitle('強度フィルタ後の類似度マップ', fontsize=13)
plt.tight_layout(rect=[0, .1, 1, .95])
for ext in ('png', 'pdf', 'svg'):
    fig.savefig(f'{FIGDIR}/fig_epitope_umap_strong.{ext}', dpi=300, bbox_inches='tight')
print('saved')

In [ ]:
# 目印化合物の共局在が改善するか（同一薬・別細胞の近傍順位、ランダム=50%）
def poscheck(E, drug):
    sub = [i for i in E.index if i[1].lower() == drug]
    if len(sub) < 2: return np.nan
    r = []
    for s in sub:
        d = np.linalg.norm(E.values - E.loc[s].values, axis=1)
        o = pd.Series(d, index=E.index).drop(s).sort_values()
        r += [(k + 1) / len(o) * 100 for k, ix in enumerate(o.index) if ix[1].lower() == drug]
    return np.median(r)

rows = []
for name in LAYERS:
    rows.append({'layer': name,
                 **{d: round(poscheck(EMB_F[name], d), 1) for d in POSCTL}})
print('強度フィルタ後の目印共局在（%、小さいほど良い / ランダム=50）')
pd.DataFrame(rows).set_index('layer')

## 7. リポジショニング表

**エピトープ空間で近い ＝ 似た糖鎖変化を起こす ＝ 転用候補**。

各アンカー薬（がん適応の承認薬）について、近傍に来た**非がん承認薬**を列挙する。
ノイズと本物を区別するため、以下を付ける：

- **再現細胞数** — そのペアが何細胞で互いに近傍に入るか（多細胞データの正しい使い道）
- 両者のシグネチャ強度
- **どのエピトープで近いのか** — 両者が同符号で大きく動かしているエピトープ上位3つ

In [ ]:
TOPK = 15

def repositioning_table(X, topk=TOPK):
    X = X.dropna(axis=1, how='any')
    Xc = X - X.groupby(level=0).transform('mean')
    mag = pd.Series(np.linalg.norm(Xc.values, axis=1), index=X.index)
    recs = []
    for cell in Xc.index.get_level_values(0).unique():
        sub = Xc.loc[cell]
        if len(sub) < topk + 1: continue
        V = sub.values
        Vn = V / np.linalg.norm(V, axis=1, keepdims=True)
        drugs = sub.index.to_numpy().astype(str)
        up = np.char.upper(drugs)
        is_onc = np.isin(up, list(onc))
        is_apr = np.isin(up, list(approved))
        cand_ok = is_apr & ~is_onc                     # 転用候補＝非がんの承認薬
        for i in np.where(is_onc)[0]:
            sim = Vn @ Vn[i]                           # 近傍は全化合物の中で計算
            sim[i] = -np.inf
            order = np.argsort(-sim)
            rank_all = np.empty(len(order), dtype=int)
            rank_all[order] = np.arange(1, len(order) + 1)
            n = 0
            for j in order:
                if not cand_ok[j]: continue            # 読み出しは承認薬のみ
                ep = pd.Series(V[i] * V[j], index=sub.columns).nlargest(3)
                recs.append({'cell': cell, 'anchor': drugs[i], 'candidate': drugs[j],
                             'cos': round(float(sim[j]), 3),
                             'rank_all': int(rank_all[j]), 'n_pool': len(order),
                             'mag_anchor': round(float(mag.loc[(cell, drugs[i])]), 2),
                             'mag_cand': round(float(mag.loc[(cell, drugs[j])]), 2),
                             'epitopes': ' / '.join(e.split('|')[1] for e in ep.index)})
                n += 1
                if n >= topk: break
    return pd.DataFrame(recs)

REP = repositioning_table(FILT['(c) epitope [経路射影]'])
# 再現細胞数
cnt = REP.groupby(['anchor', 'candidate']).size().rename('n_cells')
REP = REP.join(cnt, on=['anchor', 'candidate'])
REP['moa_cand'] = REP.candidate.str.upper().map(ann['MOA'])
REP['atc_cand'] = REP.candidate.str.upper().map(ann['ATC_L1_desc'])
REP = REP.sort_values(['n_cells', 'cos'], ascending=False)
print(f'{len(REP):,} 行  /  ユニークなペア {REP.groupby(["anchor","candidate"]).ngroups:,}')
print(f'2細胞以上で再現するペア {(cnt>=2).sum():,}  /  3細胞以上 {(cnt>=3).sum():,}')
REP.head(30)

In [ ]:
top = REP[REP.n_cells >= 3].drop_duplicates(['anchor', 'candidate'])
print(f'3細胞以上で再現する転用候補ペア: {len(top)}')
top[['anchor','candidate','n_cells','cos','moa_cand','atc_cand','epitopes']].head(40)

In [ ]:
REP.to_csv(f'{TABDIR}/epitope_repositioning_candidates.csv', index=False)
for name, E in EMB_F.items():
    E.to_csv(f'{TABDIR}/umap_{name.split()[0].strip("()")}_strong.csv')
print('saved'); con.close()

In [ ]:
for name, X in LAYERS.items():
    EMB[(name, True)].to_csv(f'{TABDIR}/umap_{name.split()[0].strip("()")}_centered.csv')
    EMB[(name, False)].to_csv(f'{TABDIR}/umap_{name.split()[0].strip("()")}_raw.csv')
nb.to_csv(f'{TABDIR}/umap_flag_neighbors.csv', index=False)
print('saved tables')

## 読むポイント

1. **(a)(b)(c) で構造が変わるか** — エピトープ射影が何かを足しているか、それとも壊しているか
2. **センタリングなしで細胞株ごとに分かれるか** — 分かれなければ前処理を疑う
3. **センタリングありで薬のATC/MoAごとに分かれるか**
4. **tunicamycin / brefeldin-a が本体から離れるか** — 離れるならエピトープ軸は糖鎖に反応している
5. **旗（系統一致の承認薬）の近傍に別細胞の何が来るか** ← 元の問い「肝のA薬と肺のB薬」の答え